# 01. Python 기준 모델과 배포 계약

**목표**

- 외부 다운로드 없이 네 개 센서 특성과 세 상태의 데이터를 만든다.
- 학습 데이터로만 전처리 통계를 계산해 데이터 누수를 막는다.
- softmax 선형 분류기를 학습하고 정확도와 p50/p95/p99 지연시간을 잰다.
- C++이 그대로 재현해야 할 입력 shape, dtype, 정규화, class 순서를 찾는다.

**실행 환경**: `cd job && source .venv/bin/activate && jupyter lab`

이 노트북은 `numpy`만 있으면 실행된다. 모델 구현의 모든 실행문은 `src/sensor_pipeline.py`에서 바로 위 한국어 주석으로 문법과 작성 이유를 설명한다.

## 1. 모듈을 불러온다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | `from ... import ...` | 모듈 전체 이름을 반복하지 않고 필요한 클래스만 현재 namespace로 가져온다. |
| 2 | 같은 import 문법 | 합성 데이터 생성 함수를 가져온다. |
| 3 | 같은 import 문법 | 정확도 계산 함수를 가져온다. |
| 4 | 같은 import 문법 | 지연시간 측정 함수를 가져온다. |

In [ ]:
# Standardizer와 SoftmaxClassifier는 전처리 통계와 학습 parameter를 각각 소유한다.
from src.sensor_pipeline import Standardizer, SoftmaxClassifier
# make_sensor_dataset은 seed가 고정된 합성 센서 입력과 정답을 만든다.
from src.sensor_pipeline import make_sensor_dataset
# accuracy는 예측과 정답이 같은 비율을 Python float로 반환한다.
from src.sensor_pipeline import accuracy
# benchmark_latency_ms는 warm-up 뒤 반복 실행의 latency 백분위를 계산한다.
from src.sensor_pipeline import benchmark_latency_ms

## 2. train과 test를 독립적으로 만든다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | tuple unpacking `a, b = ...` | 함수가 반환한 입력 배열과 정답 배열을 나눠 받는다. |
| 2 | keyword argument | test의 seed를 다르게 해 학습 때 보지 못한 noise를 만든다. |
| 3 | `.shape` | `(샘플 수, 특성 수)`와 정답 길이를 검사한다. |
| 4 | slicing `[:3]` | 전체 배열을 출력하지 않고 처음 세 건만 확인한다. |

In [ ]:
# 클래스당 120건의 학습 입력을 만들므로 전체 행은 360개다.
train_x, train_y = make_sensor_dataset(samples_per_class=120, seed=7)
# test는 다른 seed와 클래스당 40건을 사용해 독립 표본 120개를 만든다.
test_x, test_y = make_sensor_dataset(samples_per_class=40, seed=99)
# shape 출력은 모델 계약인 특성 수 4와 정답 행 수 일치를 빠르게 확인한다.
print('train:', train_x.shape, train_y.shape, 'test:', test_x.shape, test_y.shape)
# 첫 세 입력과 정답을 함께 보아 값 범위와 label 대응을 확인한다.
print(train_x[:3], train_y[:3])

## 3. 전처리는 train에만 fit한다

`test_x`까지 섞어 평균과 표준편차를 구하면 미래 평가 정보가 모델 pipeline에 들어가는 데이터 누수다.

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | classmethod `Standardizer.fit(...)` | 객체를 먼저 만들지 않고 학습 통계가 채워진 객체를 생성한다. |
| 2~3 | method call | 같은 mean/std를 train과 test에 적용한다. |
| 4 | attribute access | C++ metadata에 저장해야 할 실제 숫자를 확인한다. |

In [ ]:
# test를 전달하지 않고 학습 배열만으로 평균과 표준편차를 fit한다.
standardizer = Standardizer.fit(train_x)
# 학습 입력을 `(x - mean) / std` 식으로 표준화한다.
normalized_train_x = standardizer.transform(train_x)
# test도 반드시 학습에서 얻은 같은 mean과 std로 변환한다.
normalized_test_x = standardizer.transform(test_x)
# 이 두 배열은 모델 파일과 함께 metadata.json에 저장해야 하는 배포 계약이다.
print('mean=', standardizer.mean, 'std=', standardizer.std)

## 4. 기준 모델을 학습하고 평가한다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | constructor call | 입력 4개, 출력 3개인 가중치 행렬을 만든다. |
| 2 | `list(...)` | `fit`이 돌려준 반복 가능한 loss 기록을 indexing 가능하게 만든다. |
| 3 | method nesting | test 확률의 argmax 결과와 정답으로 정확도를 계산한다. |
| 4 | f-string | 첫/마지막 loss와 정확도를 지정한 소수 자릿수로 보여 준다. |
| 5 | slicing `[:1]` | batch 차원을 유지한 단일 샘플 shape `(1, 4)`를 만든다. |

In [ ]:
# 선형 softmax 기준 모델을 만들며 seed로 초기 가중치를 재현한다.
model = SoftmaxClassifier(input_dim=4, num_classes=3, seed=11)
# 250번의 전체 배치 gradient descent를 실행하고 epoch별 loss를 저장한다.
losses = list(model.fit(normalized_train_x, train_y, epochs=250, learning_rate=0.08))
# 보지 못한 test 데이터의 예측 클래스와 정답을 비교한다.
test_accuracy = accuracy(model.predict(normalized_test_x), test_y)
# loss가 줄고 정확도가 충분한지 사람이 확인한다.
print(f'loss {losses[0]:.4f} -> {losses[-1]:.4f}, test accuracy={test_accuracy:.4f}')
# 단일 샘플 추론을 500회 측정해 평균이 숨기는 긴 꼬리 지연시간을 본다.
print(benchmark_latency_ms(model, normalized_test_x[:1], repeats=500))

## 5. 직접 해 볼 과제

1. `shift=0.18`인 새 장치 test set을 만들고 기존 모델 정확도가 어떻게 변하는지 본다.
2. `samples_per_class=10`으로 줄여 작은 데이터에서 결과가 얼마나 흔들리는지 본다.
3. test 통계로 test를 따로 표준화하는 잘못된 실험을 해 보고 왜 제품 C++에서 재현할 수 없는지 설명한다.
4. `learning_rate`를 `0.001`, `1.0`으로 바꾸고 loss 곡선을 비교한다.
5. `python -m unittest discover -s tests -v`로 자동 회귀 테스트를 실행한다.

**통과 기준**: test 정확도 0.95 이상, 마지막 loss가 첫 loss보다 작음, 각 softmax 행의 합이 1.